In [17]:
#!pip3 install html5lib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.2/112.2 kB 1.8 MB/s eta 0:00:00a 0:00:01


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

pd.set_option('display.max_columns',1000)
pd.set_option('display.max_rows',1000)

In [2]:
import lxml
import html5lib
from urllib.request import urlopen
import time

In [3]:
df1 = pd.read_html('https://www.oddsshark.com/stats/gamelog/baseball/mlb/27024?season=2024')[0]

In [4]:
df1.head(10)

,Date,Opponent,Game,Result,Score,Line,OU,Total
0,"Mar 28, 2024",@ Oakland,REG,W,8-0,-142,O,7.5
1,"Mar 29, 2024",@ Oakland,REG,W,6-4,-127,O,8.0
2,"Mar 30, 2024",@ Oakland,REG,W,12-3,-130,O,7.5
3,"Mar 31, 2024",@ Oakland,REG,L,4-3,-119,U,8.5
4,"Apr 1, 2024",@ Seattle,REG,L,5-4,-115,O,8.0
5,"Apr 2, 2024",@ Seattle,REG,W,5-2,-102,P,7.0
6,"Apr 3, 2024",@ Seattle,REG,W,8-0,133,O,7.5
7,"Apr 4, 2024",@ Minnesota,REG,W,4-2,150,U,7.0
8,"Apr 6, 2024",@ Minnesota,REG,W,3-1,150,U,8.0
9,"Apr 8, 2024",vs Chi White Sox,REG,W,4-0,-250,U,8.5


In [6]:
def line_to_prob(line):
    prob_underdog = 100/(np.abs(line)+100) # this is the probability for the underdog
    add_term = ((1-np.sign(line))/2) # 0 if negative, 1 if positive
    mult_factor = np.sign(line) # -1 if negative, 1 if positive
    # if line is positive, team is underdog, give 0 + 1*prob_underdog
    # if line is negative, team is favorites, give 1 + (-1)*prob_underdog
    imp_prob = add_term + mult_factor * prob_underdog 
    return imp_prob

In [7]:
oddsshark_num_to_team_dict = {
    26995 + i: team for i, team in enumerate([
        'PHI', 'SDN', 'SFN', 'ANA', 'DET', 'CIN', 'NYA', 'TEX', 'TBA', 'COL',
        'MIN', 'KCA', 'ARI', 'BAL', 'ATL', 'TOR', 'SEA', 'MIL', 'PIT', 'NYN',
        'LAN', 'OAK', 'WAS', 'CHA', 'SLN', 'CHN', 'BOS', 'MIA', 'HOU', 'CLE'
    ])
}

In [8]:
for i in range(26995, 27025):
    team_name = oddsshark_num_to_team_dict[i]
    print(team_name)
    for season in range(2021,2025):
        print(season)
        url = 'https://www.oddsshark.com/stats/gamelog/baseball/mlb/'+str(i)+'?season='+str(season)
        df_temp = pd.read_html(url)[0]
        df_temp = df_temp[df_temp.Game=='REG']
        print(df_temp.shape)
        df_temp['team_source'] = team_name
        df_temp['season'] = season
        df_temp['date_numeric'] = pd.to_datetime(df_temp.Date).astype(str).str.replace('-','')
        df_temp['game_no'] = np.arange(1,df_temp.shape[0]+1)
        df_temp['prob_implied'] = line_to_prob(df_temp['Line'])      
        next_game_date = np.concatenate((df_temp['date_numeric'].iloc[1:],[0]))
        previous_game_date = np.concatenate(([0], df_temp['date_numeric'].iloc[:-1]))
        game_1_dblheader = (df_temp.date_numeric.to_numpy()==next_game_date).astype(int)
        game_2_dblheader = (df_temp.date_numeric.to_numpy()==previous_game_date).astype(int)*2
        df_temp['dblheader_num'] = game_1_dblheader+game_2_dblheader        
        fname_out = 'data/odds/oddsshark_'+team_name+'_'+str(season)+'.csv'
        df_temp.to_csv(fname_out,index=False)
        time.sleep(.1)

PHI
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
SDN
2021
(161, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
SFN
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
ANA
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
DET
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
CIN
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
NYA
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
TEX
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
TBA
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
COL
2021
(161, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
MIN
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
KCA
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
ARI
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
BAL
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
ATL
2021
(160, 8)
2022
(162, 8)
2023
(162, 8)
2024
(162, 8)
TOR
2021
(162, 8)
2022
(162, 8)
2023
(162, 8)
2024
(161, 8)
SEA
2021
(161, 8)
2022
(162, 8)
2023
(16